# Week 14 tasks (Nov 13–14, 2026)

### Summary
* **Grey bands** in the Earth-frame maps are sky SPHEREx never sees (it points ~90–115° from the Sun).
* **The "jump" near MJD 61,200** is not a release step: the tall table has none. From Apr 29 to Jun 18, 2026 the north
  ecliptic pole was visited 3× as often, many times at the noisy 4.98–5.01 μm edge.
* **SPHEREx's zodi column is Kelsall × 1.165**, with only 1–2% geometric terms; the biggest remaining term is wavelength
  (−1.4% per 0.05 μm).
* **The Week 13 behind − ahead deficit (−5% near the ecliptic) is mostly Kelsall's scale error leaking in**, because
  "behind" is seen at larger elongation. After scaling, −0.7 ± 0.4% remains (−2.6% in a joint fit): a v_e term of 1–3%
  may remain.
* **Training on real data (task 7):** 192 patches are enough to learn Kelsall (0.026% on held-out patches), but on
  SPHEREx's zodi the network ties a single scale factor (1.02% vs 0.96%), and on the measured sky it overfits
  (3.2% vs 1.9% for SPHEREx zodi + constant, off the plane).
* **Follow-ups (section 6):** with the pixel's wavelength as an input, the network reproduces SPHEREx's zodi to
  **0.20%** (3× better than the best wavelength correction). Star masking (dropping MP_SOURCE pixels) helps every model on
  the measured sky by 10–30%, but the network (2.3%) still trails SPHEREx zodi + constant (1.7%) off the plane.


In [ ]:
const R = joinpath(@__DIR__, "..")
struct PNGFile; path::String; end; Base.show(io::IO, ::MIME"image/png", p::PNGFile) = write(io, read(p.path))   # raw PNG bytes: renders in any Jupyter front end
showpng(f) = PNGFile(joinpath(R, "figures", f))


## 1. What are the grey bands?
Our model's error on withheld months in the frame of Earth's motion (centre = behind Earth), labelled. A and B are the
Sun-avoidance and anti-Sun limits, C is outside the ellipse, D is cells with fewer than 30 exposures.


In [ ]:
showpng("11132026_earth2d_gray_annot.png")


## 2. The MJD 61,200 "jump"
Measured − SPHEREx zodi at the north ecliptic pole against time (orange: 4.98–5.01 μm), and exposures per 5 days.
The level stays at ~0.026 MJy/sr; the scatter doubles at the detector edge (0.010 vs 0.0055 MJy/sr).


In [ ]:
showpng("11132026_nep_burst.png")


No release step in the tall table: SPHEREx zodi / ZodiPy against time, coloured by release.


In [ ]:
showpng("11132026_release_jump_L6_grid2.png")


## 3. What model is SPHEREx's zodi column?
SPHEREx zodi / ZodiPy for 8938 exposures in 192 patches. Median 1.165 (16–84%: 1.147–1.186). A regression on wavelength,
elongation, |β|, side of Earth and season leaves 1.03% of the 1.69% rms.


In [ ]:
showpng("11132026_spherex_vs_kelsall.png")


## 4. The behind − ahead leak
Per patch, behind − ahead in the median-subtracted residual, before and after scaling our model by 1.181, and the
elongations at which each side is seen (109° behind vs 93° ahead near the ecliptic).

| Behind − ahead (|β| < 30°, 74 patches) | As in Week 13 | Scale fixed first | Joint fit |
|---|---|---|---|
| Measured − our model | −5.0 ± 0.3% | −0.7 ± 0.4% | −2.6 ± 0.2% |
| Measured − SPHEREx zodi | −0.1 ± 0.3% | — | −1.4 ± 0.2% |


In [ ]:
showpng("11132026_residual_leak.png")


## 5. The emulator trained on real data (task 7)
**Training set (7a):** Discovery extracted per-exposure medians in level-9 sub-pixels of the 192 grid-2 patches at
4.9–5.0 μm (`realdata/talltable_extract.py`, `ttx_merge.py`): 197,523 rows, 8938 exposures. ZodiPy was evaluated at
every row (`realdata/ttx_zodipy.py`, times sorted, see the choices log for the ZodiPy 1.1.5 batching bug), and our
inputs taken from the nearest simulated exposure (median 1 min apart; `realdata/ttx_ours.jl`). Our emulator matches
ZodiPy on these rows to 0.016% (median), 0.055% (p99).

**Runs (7b):** `sweeps/train_real.jl` with the best simulation architecture (parallax band layer, sun-season inputs,
cos×exp schedule, 1000 epochs). Targets at 5.0 μm via Kelsall's colour: `zodi5` = SPHEREx zodi, `flux5` = measured sky,
`kel5` = ZodiPy (control). Fold f holds out patches with index % 5 == f. 5 seeds on fold 0 (all three targets), 2 seeds
on folds 1–4 (zodi5, flux5): 31 runs on Colibrí (jobs 211, 222; ~23 min, 5 GB each).

The report below averages the seeds and compares with baselines fit on the training patches only
(`sweeps/report_real.jl`; needs the git-ignored `data/tt_train/` tables and `sweeps/results_real/` predictions).


In [ ]:
include(joinpath(R, "sweeps", "report_real.jl"))


| Trained on | Network, median \|Δ\| (rms) | Best simple baseline, median \|Δ\| (rms) |
|---|---|---|
| ZodiPy (Kelsall), fold 0 | 0.026% (0.069%) | our emulator: 0.011% (0.017%) |
| SPHEREx zodi, all folds | 1.02% (2.03%) | Kelsall × a: 0.96% (1.45%) |
| Measured sky, \|b\| > 15°, all folds | 3.2% (13.8%) | SPHEREx zodi + b: 1.9% (4.1%) |

* The control shows the patch coverage is enough: the network learns Kelsall on unseen patches to 0.026%.
* On SPHEREx's zodi the network ties a single scale factor. It has no wavelength input, and SPHEREx zodi / Kelsall
  changes by −1.4% per 0.05 μm; a wavelength-only linear fit alone would take the spread from 0.96% to 0.65%.
* On the measured sky it fits training patches to 1.2% but held-out ones to 4%: it learns the stars and residual sky of
  the training patches (no masking yet).


In [ ]:
showpng("11142026_real_maps.png")


* Top row: the network without wavelength removes the smooth ecliptic pattern of Kelsall × 1.164 but adds 1–3% misses in
  single patches; per-patch medians tie (0.43% vs 0.44%).
* Bottom row: see section 6.

## 6. Wavelength as an input, and the star mask (Nov 14, later)
The user approved both. **Wavelength:** the row's wavelength (4.9–5.01 μm) is one more input, placed before the band
layer's 6 inputs (`zodi5L`; Colibrí job 232, 10 runs). **Star mask:** the tall table's default query already drops the
artifact flags; Discovery job 700324 re-extracted the medians without MP_SOURCE pixels (bit 21, pixels on catalog
sources; `TTX_MASK=1 realdata/talltable_extract.py`, `realdata/ttx_mask_merge.py`). 59% of pixels kept (73% at
|b| > 15°, 13% on the plane). Then `flux5m` and `flux5mL` (masked, without / with wavelength; job 243, 20 runs).
The report above already includes them (all 5 folds); summary:

| Target, held out | Network | Network + λ | Best baseline |
|---|---|---|---|
| SPHEREx zodi | 1.02% (2.03%) | **0.20% (0.28%)** | Kelsall × f(λ) 0.62% (0.95%) |
| Measured sky, \|b\| > 15°, unmasked | 3.2% (13.8%) | — | SPHEREx zodi + b 1.9% (4.1%) |
| Measured sky, \|b\| > 15°, star-masked | 3.0% (18.7%) | 2.3% (12.2%) | SPHEREx zodi + b 1.7% (3.2%) |

* **With the wavelength input the network reproduces SPHEREx's zodi to 0.20%**, 3× better than the best wavelength-only
  correction: it learns how SPHEREx's model differs from Kelsall with both wavelength and geometry. Per-patch medians
  0.04% (bottom-left map).
* **The star mask helps every model by 10–30%** on the measured sky, but the network still doesn't beat SPHEREx's zodi + a
  constant. Its misses are a few patches it gets badly wrong (8% of rows off by > 10%, vs 1.2% for the baseline), mostly
  near the Galactic plane. The same inputs work for SPHEREx's zodi, so the noisy target is the problem.

## 7. A robust loss and clean rows only (Nov 14, afternoon)
The user picked two fixes: a relative Huber loss (quadratic below 2% relative error, linear above; `target=:relhuber`
in `sweeps/mb_train.jl`) and training only on rows that keep ≥ 50% of their pixels after the star mask
(`train_real.jl ... huber 0.5`). Colibrí job 254: flux5mL with both, and with the cut only (MSE), 2 seeds × 5 folds each.

Star-masked measured sky, |b| > 15°, rows ≥ 50% clean (the same 141,387 rows for every line):

| Model | Median \|Δ\| | rms | Rows off by > 10% |
|---|---|---|---|
| Network + λ (all masked rows, MSE) | 2.16% | 8.85% | 6.5% |
| Network + λ, ≥ 50%-clean rows (MSE) | 1.29% | 4.46% | 2.2% |
| **Network + λ, ≥ 50%-clean rows, Huber** | **1.20%** | **2.77%** | **0.8%** |
| SPHEREx zodi + constant | 1.44% | 2.63% | — |

* **The network now predicts the measured sky better than SPHEREx's zodi model plus a constant** (median), with about
  the same rms. The clean-row cut gives most of the gain; the Huber loss removes most of the remaining outliers.
* The cut drops 27% of rows, mostly on the Galactic plane, so this holds for the sky away from the plane.
* Bottom-right map: the Huber + cut network (blank patches have no rows left).
